In [9]:
import cv2 as cv
import numpy as np
import os
from video_to_frames import video_to_frames

**Parameters**

In [19]:
# Frame params
FRAME_HEIGHT = 480
FRAME_WIDTH = 640

num_bins_x = 3
num_bins_y = 1

# Keypoint params
response_cutoff = 0.000

# Number of frames to process
n_frames = 25

In [20]:
# Create output folder
output_folder = "orb_keypoints"
os.makedirs(output_folder, exist_ok=True)

# Initiate ORB detector
orb = cv.ORB_create(nfeatures=500)

# Storing keypoint data
kp_list = []

# Process images
for i in range(1, n_frames + 1):
    img_path = rf"Images\img{i:04d}.png"
    
    img = cv.imread(img_path, cv.IMREAD_GRAYSCALE)
    
    if img is None:
        print(f"Failed to load {img_path}")
        continue
    
    # Detect and compute keypoints
    kp = orb.detect(img, None)
    kp, des = orb.compute(img, kp)

    # Filter keypoints by response
    kp_filtered = [k for k in kp if k.response >= response_cutoff]

    # Save raw keypoints
    kp_list.append(kp_filtered)
    
    # Draw keypoints
    img_kp = cv.drawKeypoints(img, kp_filtered, None, color=(0,255,0), flags=0)
    
    # Draw grid
    bin_width = FRAME_WIDTH // num_bins_x
    bin_height = FRAME_HEIGHT // num_bins_y
    
    # Vertical lines
    for x in range(0, FRAME_WIDTH + 1, bin_width):
        cv.line(img_kp, (x, 0), (x, FRAME_HEIGHT), (0, 0, 255), 2)
    
    # Horizontal lines
    for y in range(0, FRAME_HEIGHT + 1, bin_height):
        cv.line(img_kp, (0, y), (FRAME_WIDTH, y), (0, 0, 255), 2)
    
    # Save result
    output_path = os.path.join(output_folder, f'kp_img{i:04d}.png')
    cv.imwrite(output_path, img_kp)

print(f"\nAll images saved to: {output_folder}")


All images saved to: orb_keypoints


**Bin point count**

In [13]:
x_data = []
y_data = []
for frame in kp_list:
    x_data.extend([kp.pt[0] for kp in frame if kp.response >= response_cutoff])
    y_data.extend([kp.pt[1] for kp in frame if kp.response >= response_cutoff])

hist, x_edges, y_edges = np.histogram2d(x=x_data, y=y_data, bins=[num_bins_x, num_bins_y], range=[[0, FRAME_HEIGHT], [0, FRAME_WIDTH]])

for i in range(len(x_edges) - 1):
    for j in range(len(y_edges) - 1):
        print(f"Point count [[{x_edges[i]:.2f}, {x_edges[i+1]:.2f}], [{y_edges[j]:.2f}, {y_edges[j+1]:.2f}]]: {hist[i, j]}")


Point count [[0.00, 240.00], [0.00, 640.00]]: 8299.0
Point count [[240.00, 480.00], [0.00, 640.00]]: 7467.0


In [14]:

for frame_idx, keypoints in enumerate(kp_list, start=1):
    print(f"\n--- Frame {frame_idx} ---")
    for kp in keypoints:
        if kp.response >= 0.001:
            print(f"  xy: ({kp.pt[0]:.2f}, {kp.pt[1]:.2f}) -- response: {kp.response:.3f}")


--- Frame 1 ---

--- Frame 2 ---
  xy: (91.00, 48.00) -- response: 0.001
  xy: (91.00, 50.00) -- response: 0.001

--- Frame 3 ---
  xy: (91.00, 49.00) -- response: 0.001

--- Frame 4 ---

--- Frame 5 ---
  xy: (100.00, 50.00) -- response: 0.001
  xy: (60.00, 165.00) -- response: 0.001

--- Frame 6 ---
  xy: (103.00, 49.00) -- response: 0.001
  xy: (105.00, 50.00) -- response: 0.001
  xy: (65.00, 165.00) -- response: 0.001

--- Frame 7 ---
  xy: (111.00, 49.00) -- response: 0.002
  xy: (113.00, 49.00) -- response: 0.001
  xy: (111.00, 52.00) -- response: 0.001
  xy: (36.00, 149.00) -- response: 0.001
  xy: (112.80, 49.20) -- response: 0.001
  xy: (112.80, 51.60) -- response: 0.001

--- Frame 8 ---
  xy: (122.00, 52.00) -- response: 0.001
  xy: (45.00, 150.00) -- response: 0.001

--- Frame 9 ---

--- Frame 10 ---

--- Frame 11 ---
  xy: (160.00, 56.00) -- response: 0.001

--- Frame 12 ---

--- Frame 13 ---
  xy: (193.00, 55.00) -- response: 0.001
  xy: (193.00, 57.00) -- response: 0.001